In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
if os.environ.get("GEMINI_API_KEY"):
    print("GEMINI_API_KEY is set in the environment.")

GEMINI_API_KEY is set in the environment.


In [3]:
from langchain_openai import ChatOpenAI

LLM = ChatOpenAI(
    # model_name="gpt-4o",
    model_name="gemini-3.5-flash-lite",
    openai_api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [25]:
# response = LLM.invoke("what is AI Agents, tell me in 1 paragraph?")
# response.content

## **RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA**

## **Step 1: Preparing documents for your text data**

In [4]:
from langchain_core.documents import Document

my_text = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g. images, audio, and videos).

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]

Artificial intelligence was founded as an academic discipline in 1956.[4] The field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Funding and interest increased substantially after 2012, when graphics processing units (GPUs) started being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[9] This growth accelerated further after 2017 with the transformer architecture.[10] In the 2020s, an AI boom coincided with advances in generative AI, which became widespread and allowed for the creation and modification of media. In addition to AI safety and unintended consequences and harms from the use of AI, ethical concerns, AI's long-term effects, environmental effects,[11] and potential existential risks have prompted discussions of AI regulation."""

document = Document(page_content=my_text, metadata={"source": "my_text.txt", "author": "Akash"})

## STEP2: Splitting the documents into chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents([document])
chunks

[Document(metadata={'source': 'my_text.txt', 'author': 'Akash'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'my_text.txt', 'author': 'Akash'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g. images, audio, and videos).'),
 Document(metadata={'source': 'my_text.txt', 'author': 'Akash'}, page_content='The traditional goals of 

## **STEP3: Creating Embeddings for the chunks**

In [6]:
# from langchain_openai import OpenAIEmbeddings

# embedding_model = OpenAIEmbeddings(
#     model="text-embedding-5-small")

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    # google_api_key="your_key_here"  # Optional: Or use the GOOGLE_API_KEY environment variable
)
text = "Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]"
query_result = embedding_model.embed_query(text)

print(f"Embedding length: {len(query_result)}")

Embedding length: 3072


## Step4: Create and Store Embedding Vectors

In [7]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_db"
    )


C:\Users\Dell\AppData\Local\Temp\ipykernel_14744\1833045862.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


## STEP5: Semantic Search

In [34]:
context = vectorstore.similarity_search("What is AI?", k=3)


In [37]:
response = LLM.invoke("What is AI?,You can use the following context to answer the question: " + str(context))
print(response.content)

Based on the provided context, **Artificial Intelligence (AI)** is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. 

Additionally, it is a field of research within engineering, mathematics, and computer science that focuses on developing and studying methods and software to help machines perceive their environment, learn, and take actions to maximize their chances of achieving specific goals.
